In [1]:
import os
import numpy as np
import warnings
from ase.build import molecule
from ase.calculators.nwchem import NWChem
from ase.optimize import BFGS

warnings.filterwarnings('ignore')

sim_path = os.path.expanduser('~/QE/theory_scan')
os.makedirs(sim_path, exist_ok=True)

## Scanning theories and basis-sets

Goal: find the best theory + basis-set pair for methane geometry optimization.

- **Theories scanned:** Hartree-Fock, MP2, DFT-B3LYP  (fixed basis: cc-pVTZ)
- **Basis-sets scanned:** STO-3G, cc-pVDZ, cc-pVTZ  (fixed theory: DFT-B3LYP)
- **Reference C-H bond length:** 1.094 Å (experimental)

In [2]:
theories = {
    'Hartree-Fock': 'scf',
    'MP2':          'mp2',
    'DFT-B3LYP':   'dft',
}

basis_sets = {
    'STO-3G':  'sto-3g',
    'cc-pVDZ': 'cc-pvdz',
    'cc-pVTZ': 'cc-pvtz',
}

In [3]:
def create_calculator(theory, basis_set):

    folder_name = f'T_{theory}_B_{basis_set}'.replace('-', '_')
    os.makedirs(os.path.join(sim_path, folder_name), exist_ok=True)
    clean_label = os.path.join(sim_path, folder_name, 'calc')

    kwargs = {
        'label':  clean_label,
        'theory': theory,
        'basis':  basis_set,

    }


    if theory == 'dft':
        kwargs['dft'] = {'xc': 'b3lyp'}

    return NWChem(**kwargs)

In [4]:
theories_variation = {
    'Theory type':        [],
    'Final bond length':  [],
    'Relative error (%)': [],
}

basis_set_variation = {
    'Basis set type':        [],
    'Final bond length':  [],
    'Relative error (%)': [],
}

In [5]:
ref_bond = 1.094   

print('='*55)
for i, (name, theory) in enumerate(theories.items()):
    print(f'[{i+1}/{len(theories)}] {name}')

    methane = molecule('CH4')  
    methane.calc = create_calculator(theory, basis_sets['cc-pVTZ'])

    opt = BFGS(methane, logfile=None)
    opt.run(fmax=0.05)


    coords = methane.get_positions()
    bonds  = [np.linalg.norm(coords[0] - coords[i]) for i in range(1, 5)]
    mean_bond = np.mean(bonds)
    rel_error = abs(mean_bond - ref_bond) / ref_bond * 100

    theories_variation['Theory type'].append(name)
    theories_variation['Final bond length'].append(round(mean_bond, 4))
    theories_variation['Relative error (%)'].append(round(rel_error, 2))

    print(f'  Bond length   : {mean_bond:.4f} Å')
    print(f'  Relative error: {rel_error:.2f}%\n')

print('='*55)
print(f'{"Theory":<16}  {"C-H (Å)":>8}  {"Error (%)":>10}')
print('-'*40)
for n, b, e in zip(theories_variation['Theory type'],
                   theories_variation['Final bond length'],
                   theories_variation['Relative error (%)']):
    print(f'{n:<16}  {b:>8.4f}  {e:>10.2f}')
print('='*55)

[1/3] Hartree-Fock


[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations


  Bond length   : 1.0820 Å
  Relative error: 1.10%

[2/3] MP2


[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations


  Bond length   : 1.0794 Å
  Relative error: 1.34%

[3/3] DFT-B3LYP


[0] ARMCI Warning: Freed 1 leaked allocations


  Bond length   : 1.0889 Å
  Relative error: 0.46%

Theory             C-H (Å)   Error (%)
----------------------------------------
Hartree-Fock        1.0820        1.10
MP2                 1.0794        1.34
DFT-B3LYP           1.0889        0.46


[0] ARMCI Warning: Freed 1 leaked allocations


## Theory scan result → DFT-B3LYP wins

DFT-B3LYP gave the lowest relative error (0.46%) against the experimental bond length.
HF and MP2 both overshot it.

Next: fix the theory to **DFT-B3LYP** and scan across basis-sets.

In [6]:
ref_bond = 1.094   

print('='*55)
for i, (name, basis_set) in enumerate(basis_sets.items()):
    print(f'[{i+1}/{len(theories)}] {name}')

    methane = molecule('CH4')  
    methane.calc = create_calculator('dft',basis_set)

    opt = BFGS(methane, logfile=None)
    opt.run(fmax=0.05)


    coords = methane.get_positions()
    bonds  = [np.linalg.norm(coords[0] - coords[i]) for i in range(1, 5)]
    mean_bond = np.mean(bonds)
    rel_error = abs(mean_bond - ref_bond) / ref_bond * 100

    basis_set_variation['Basis set type'].append(name)
    basis_set_variation['Relative error (%)'].append(round(rel_error, 2))
    basis_set_variation['Final bond length'].append(round(mean_bond, 4))

    print(f'  Bond length   : {mean_bond:.4f} Å')
    print(f'  Relative error: {rel_error:.2f}%\n')

print('='*55)
print(f'{"Basis set":<16}  {"C-H (Å)":>8}  {"Error (%)":>10}')
print('-'*40)
for n, b, e in zip(basis_set_variation['Basis set type'],
                   basis_set_variation['Final bond length'],
                   basis_set_variation['Relative error (%)']):
    print(f'{n:<16}  {b:>8.4f}  {e:>10.2f}')
print('='*55)

[1/3] STO-3G


[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations


  Bond length   : 1.0968 Å
  Relative error: 0.25%

[2/3] cc-pVDZ


[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations
[0] ARMCI Warning: Freed 1 leaked allocations


  Bond length   : 1.0994 Å
  Relative error: 0.50%

[3/3] cc-pVTZ


[0] ARMCI Warning: Freed 1 leaked allocations


  Bond length   : 1.0889 Å
  Relative error: 0.46%

Basis set          C-H (Å)   Error (%)
----------------------------------------
STO-3G              1.0968        0.25
cc-pVDZ             1.0994        0.50
cc-pVTZ             1.0889        0.46


[0] ARMCI Warning: Freed 1 leaked allocations


## Final verdict

For methane geometry optimization:

| Parameter | Choice     | Error |
|-----------|------------|-------|
| Theory    | DFT-B3LYP  | 0.46% |
| Basis set | STO-3G     | 0.25% |

STO-3G is the smallest and fastest basis-set here, yet gives the best accuracy — a good trade-off for structure optimization.